In [1]:
import pandas as pd
import pyarrow.parquet as pq

# AUDIT 1 — COMPAR:IA conversations
conv = pd.read_parquet('../data/raw/conversations.parquet')
print("=== CONVERSATIONS ===")
print(f"Lignes : {len(conv)}")
print(f"Colonnes : {conv.shape[1]}")
print("\nColonnes disponibles :")
print(conv.columns.tolist())
print("\nAperçu types :")
print(conv.dtypes)
print("\nValeurs nulles (%) :")
print((conv.isnull().sum() / len(conv) * 100).round(1))

=== CONVERSATIONS ===
Lignes : 18690
Colonnes : 30

Colonnes disponibles :
['id', 'timestamp', 'session_hash', 'visitor_id', 'mode', 'custom_models_selection', 'conv_turns', 'conversation_pair_id', 'model_pair_name', 'opening_msg', 'model_a_name', 'model_b_name', 'conv_a_id', 'conv_b_id', 'system_prompt_a', 'system_prompt_b', 'conversation_a', 'conversation_b', 'total_conv_a_output_tokens', 'total_conv_b_output_tokens', 'short_summary', 'keywords', 'categories', 'languages', 'model_a_total_params', 'model_b_total_params', 'model_a_active_params', 'model_b_active_params', 'total_conv_a_kwh', 'total_conv_b_kwh']

Aperçu types :
id                                     int64
timestamp                     datetime64[us]
session_hash                          object
visitor_id                            object
mode                                  object
custom_models_selection               object
conv_turns                             int64
conversation_pair_id                  object
model_

In [2]:
# ============================================================
# AUDIT 2 — COMPAR:IA reactions
# ============================================================
reactions = pd.read_parquet('../data/raw/reactions.parquet')
print("=== REACTIONS ===")
print(f"Lignes : {len(reactions)}")
print(f"Colonnes : {reactions.shape[1]}")
print("\nValeurs nulles (%) :")
print((reactions.isnull().sum() / len(reactions) * 100).round(1))
print("\nDistribution liked/disliked :")
print(reactions[['liked','disliked']].value_counts())

=== REACTIONS ===
Lignes : 88501
Colonnes : 34

Valeurs nulles (%) :
id                                  0.0
timestamp                           0.0
session_hash                        0.0
conversation_pair_id                0.0
current_conv_turn_when_reacting     0.0
model_pos                           0.0
refers_to_model                     0.0
refers_to_conv_id                   0.0
system_prompt                      65.2
response_content                    0.0
question_content                    0.0
msg_index                           0.0
msg_rank                            0.0
question_id                         0.0
liked                               0.0
disliked                            0.0
comment                             1.0
useful                              0.6
complete                            0.0
creative                            0.6
clear_formatting                    0.6
incorrect                           0.6
superficial                         0.6
instruction

In [3]:
# ============================================================
# AUDIT 2bis — COMPAR:IA votes
# ============================================================
votes = pd.read_parquet('../data/raw/votes.parquet')
print("=== VOTES ===")
print(f"Lignes : {len(votes)}")
print(f"Colonnes : {votes.shape[1]}")
print("\nColonnes :")
print(votes.columns.tolist())
print("\nValeurs nulles (%) :")
print((votes.isnull().sum() / len(votes) * 100).round(1))
print("\nDistribution chosen_model :")
print(votes['chosen_model_name'].value_counts().head(20))

=== VOTES ===
Lignes : 139935
Colonnes : 34

Colonnes :
['id', 'timestamp', 'chosen_model_name', 'both_equal', 'conversation_pair_id', 'session_hash', 'conv_comments_a', 'conv_comments_b', 'conv_useful_a', 'conv_useful_b', 'conv_creative_a', 'conv_creative_b', 'conv_clear_formatting_a', 'conv_clear_formatting_b', 'conv_incorrect_a', 'conv_incorrect_b', 'conv_superficial_a', 'conv_superficial_b', 'conv_instructions_not_followed_a', 'conv_instructions_not_followed_b', 'conv_complete_a', 'conv_complete_b', 'archived_reason', 'archived_at', 'visitor_id', 'conv_turns', 'model_pair_name', 'opening_msg', 'model_a_name', 'model_b_name', 'system_prompt_a', 'system_prompt_b', 'conversation_a', 'conversation_b']

Valeurs nulles (%) :
id                                    0.0
timestamp                             0.0
chosen_model_name                    31.8
both_equal                            6.6
conversation_pair_id                  0.0
session_hash                          0.0
conv_comments_a

In [4]:
# ============================================================
# AUDIT 4 — FMTI Stanford
# ============================================================
scores = pd.read_csv('../data/raw/Dec2025_scores (1).csv')
indicators = pd.read_csv('../data/raw/Dec2025_indicators.csv')

print("=== SCORE GLOBAL PAR FOURNISSEUR ===")
score_global = scores.iloc[:, 1:].sum().sort_values(ascending=False)
print(score_global)

print("\n=== DOMAINES ===")
print(indicators['Domain'].value_counts())

print("\n=== SOUS-DOMAINES ===")
print(indicators['Subdomain'].value_counts())

=== SCORE GLOBAL PAR FOURNISSEUR ===
IBM           95
Writer        72
AI21 Labs     66
Anthropic     46
Google        41
Amazon        39
OpenAI        35
DeepSeek      32
Meta          31
Alibaba       26
Mistral       18
Midjourney    14
xAI           14
dtype: int64

=== DOMAINES ===
Domain
Downstream    36
Upstream      34
Model         30
Name: count, dtype: int64

=== SOUS-DOMAINES ===
Subdomain
Data Acquisition              12
Compute                        9
Release                        8
Impact                         7
Post-deployment monitoring     7
Data Properties                5
Downstream mitigations         5
Risks                          5
Model Mitigations              5
Usage data                     5
Acceptable use policy          5
Model access                   4
Capabilities                   4
Model Behavior Policy          4
Model information              4
Data Processing                3
Methods                        3
Other resources                2


In [5]:
# ============================================================
# STRUCTURE FMTI
# ============================================================
print("=== SCORES (extrait) ===")
print(scores.head(5).to_string())

print("\n=== INDICATORS (extrait) ===")
print(indicators[['Domain', 'Subdomain', 'Indicator']].head(10).to_string())

print("\n=== FOURNISSEURS DISPONIBLES ===")
print(scores.columns.tolist())

print("\n=== SCORES PAR DOMAINE PAR FOURNISSEUR ===")
# Fusionner scores avec domaines
merged = scores.merge(indicators[['Indicator', 'Domain']], on='Indicator')
par_domaine = merged.groupby('Domain').sum(numeric_only=True)
print(par_domaine.to_string())

=== SCORES (extrait) ===
                               Indicator  AI21 Labs  Alibaba  Amazon  Anthropic  DeepSeek  Google  IBM  Meta  Midjourney  Mistral  OpenAI  Writer  xAI
0               Data acquisition methods          1        0       1          1         0       0    1     1           0        0       0       1    0
1                        Public datasets          1        0       0          0         0       0    1     0           0        0       0       0    0
2                               Crawling          1        0       0          1         0       1    1     1           0        0       1       0    0
3            Usage data used in training          1        0       0          0         0       0    1     1           0        0       0       1    0
4  Notice of usage data used in training          1        0       0          1         0       1    1     1           0        0       0       1    0

=== INDICATORS (extrait) ===
     Domain         Subdomain          

In [6]:
# ============================================================
# STRUCTURE indicators en détail
# ============================================================
print("=== TOUTES LES COLONNES ===")
print(indicators.columns.tolist())

print("\n=== EXTRAIT COMPLET (5 lignes) ===")
print(indicators.head(5).to_string())

print("\n=== NB INDICATEURS PAR SOUS-DOMAINE ===")
print(indicators.groupby(['Domain', 'Subdomain']).size().to_string())

=== TOUTES LES COLONNES ===
['Domain', 'Subdomain', 'Indicator', 'Definition', 'Notes', 'Example disclosure']

=== EXTRAIT COMPLET (5 lignes) ===
     Domain         Subdomain                              Indicator                                                                                                                                   Definition                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [7]:
# ============================================================
# AUDIT 5 — BASE CARBONE ADEME
# ============================================================
ademe = pd.read_csv('../data/raw/Base_Carbone_V23.6.csv', 
                    encoding='latin1', sep=';', low_memory=False)

mask = (
    ademe['Nom base français'].astype(str).str.lower().str.contains('électricité|electricit', na=False)
) & (
    ademe["Statut de l'élément"] == 'Valide générique'
) & (
    ademe['Nom attribut français'].astype(str).str.contains('mix moyen', na=False)
)

elec = ademe[mask][[
    'Sous-localisation géographique français',
    'Total poste non décomposé',
    'Unité français'
]].rename(columns={
    'Sous-localisation géographique français': 'pays',
    'Total poste non décomposé': 'facteur_kgco2_kwh'
})

print(f"Pays disponibles : {len(elec)}")
print(elec.to_string())

Pays disponibles : 254
                                   pays facteur_kgco2_kwh Unité français
5049                                NaN            0,0785     kgCO2e/kWh
5050                                NaN            0,0569     kgCO2e/kWh
5051                                NaN            0,0129     kgCO2e/kWh
5052                                NaN          8,70E-03     kgCO2e/kWh
5132                                NaN            0,0785     kgCO2e/kWh
5133                                NaN            0,0569     kgCO2e/kWh
5134                                NaN            0,0129     kgCO2e/kWh
5135                                NaN          8,70E-03     kgCO2e/kWh
5212                                NaN            0,0785     kgCO2e/kWh
5213                                NaN            0,0569     kgCO2e/kWh
5214                                NaN            0,0129     kgCO2e/kWh
5215                                NaN          8,70E-03     kgCO2e/kWh
5292                        

In [9]:
# Debug — voir les valeurs de Type poste pour les lignes électricité
mask2 = (
    ademe['Nom base français'].astype(str).str.lower().str.contains('électricité|electricit', na=False)
) & (
    ademe["Statut de l'élément"] == 'Valide générique'
) & (
    ademe['Nom attribut français'].astype(str).str.contains('mix moyen', na=False)
)

print(ademe[mask2]['Type poste'].value_counts())
print("\nNom poste français :")
print(ademe[mask2]['Nom poste français'].value_counts().head(10))

Type poste
Combustion à la centrale     29
Amont                        29
Transport et distribution    29
Emissions fugitives           1
Name: count, dtype: int64

Nom poste français :
Nom poste français
Pertes                                              19
Amont des combustibles et amortissement centrale     6
Perte SF6 des transformateurs                        4
barrage de Petit Saut                                1
Name: count, dtype: int64


In [10]:
# ============================================================
# ADEME — somme des postes par pays
# ============================================================
mask3 = (
    ademe['Nom base français'].astype(str).str.lower().str.contains('électricité|electricit', na=False)
) & (
    ademe["Statut de l'élément"] == 'Valide générique'
) & (
    ademe['Nom attribut français'].astype(str).str.contains('mix moyen', na=False)
)

elec = ademe[mask3][[
    'Sous-localisation géographique français',
    'Total poste non décomposé'
]].rename(columns={
    'Sous-localisation géographique français': 'pays',
    'Total poste non décomposé': 'facteur_kgco2_kwh'
})

# Convertir les virgules en points
elec['facteur_kgco2_kwh'] = elec['facteur_kgco2_kwh'].astype(str).str.replace(',', '.').str.strip()
elec['facteur_kgco2_kwh'] = pd.to_numeric(elec['facteur_kgco2_kwh'], errors='coerce')

# Sommer les postes par pays
elec_grouped = elec.groupby('pays')['facteur_kgco2_kwh'].sum().reset_index()
elec_grouped = elec_grouped.sort_values('facteur_kgco2_kwh')

print(f"Pays disponibles : {len(elec_grouped)}")
print(elec_grouped.to_string())

Pays disponibles : 149
                                  pays  facteur_kgco2_kwh
62                             Islande           0.000183
88                          Mozambique           0.000648
94                               Népal           0.001060
1                              Albanie           0.002150
141                             Zambie           0.002680
109                 Rép. Dém. Du Congo           0.002910
148                           Éthiopie           0.007010
125                        Tadjikistan           0.014300
92                             Norvège           0.016700
121                             Suisse           0.027300
122                              Suède           0.029600
35                          Costa rica           0.055700
70                        Kirghizistan           0.059100
54                             Géorgie           0.068700
45                              France           0.079100
137                            Uruguay           

In [11]:
# ============================================================
# AUDIT NOMS DE MODÈLES — avant nettoyage
# ============================================================
tous_modeles = pd.concat([
    conv['model_a_name'],
    conv['model_b_name'],
    votes['model_a_name'],
    votes['model_b_name'],
    reactions['model_a_name'],
    reactions['model_b_name']
]).unique()

tous_modeles_sorted = sorted([m for m in tous_modeles if pd.notna(m)])
print(f"Nb modèles uniques : {len(tous_modeles_sorted)}")
for m in tous_modeles_sorted:
    print(m)

Nb modèles uniques : 116
Apertus-70B-Instruct-2509
Apertus-8B-Instruct-2509
DeepSeek-V3.2
EuroLLM-22B-Instruct-2512
Qwen3-Coder-480B-A35B-Instruct
Yi-1.5-9B-Chat
aya-expanse-32b
aya-expanse-8b
c4ai-command-r-08-2024
chocolatine-14b-instruct-dpo-v1.2-q4
chocolatine-2-14b-instruct-v2.0.3-q8
claude-3-5-sonnet-v2
claude-3-7-sonnet
claude-4-5-sonnet
claude-4-6-sonnet
claude-4-sonnet
command-a
deepseek-chat-v3.1
deepseek-r1
deepseek-r1-0528
deepseek-r1-distill-llama-70b
deepseek-v3-0324
deepseek-v3-chat
gemini-1.5-pro
gemini-2.0-flash
gemini-2.5-flash
gemini-3-flash-preview
gemini-3-pro-preview
gemini-3.1-flash-lite-preview
gemini-3.1-pro-preview
gemma-2-27b-it-q8
gemma-2-9b-it
gemma-3-12b
gemma-3-27b
gemma-3-4b
gemma-3n-e4b-it
gemma-4-26b-a4b-it
gemma-4-31b-it
glm-4.5
glm-4.6
glm-4.7
glm-5
glm-5.1
gpt-4.1-mini
gpt-4.1-nano
gpt-4o-2024-08-06
gpt-4o-mini-2024-07-18
gpt-5
gpt-5-mini
gpt-5-nano
gpt-5.1
gpt-5.2
gpt-5.3
gpt-5.4
gpt-5.4-mini
gpt-5.4-nano
gpt-oss-120b
gpt-oss-20b
grok-3-mini-beta
g

In [12]:
# ============================================================
# MAPPING MODÈLE → FOURNISSEUR
# ============================================================
mapping_fournisseur = {
    # Anthropic
    'claude-3-5-sonnet-v2': 'Anthropic',
    'claude-3-7-sonnet': 'Anthropic',
    'claude-4-5-sonnet': 'Anthropic',
    'claude-4-6-sonnet': 'Anthropic',
    'claude-4-sonnet': 'Anthropic',
    # OpenAI
    'gpt-4.1-mini': 'OpenAI',
    'gpt-4.1-nano': 'OpenAI',
    'gpt-4o-2024-08-06': 'OpenAI',
    'gpt-4o-mini-2024-07-18': 'OpenAI',
    'gpt-5': 'OpenAI',
    'gpt-5-mini': 'OpenAI',
    'gpt-5-nano': 'OpenAI',
    'gpt-5.1': 'OpenAI',
    'gpt-5.2': 'OpenAI',
    'gpt-5.3': 'OpenAI',
    'gpt-5.4': 'OpenAI',
    'gpt-5.4-mini': 'OpenAI',
    'gpt-5.4-nano': 'OpenAI',
    'gpt-oss-120b': 'OpenAI',
    'gpt-oss-20b': 'OpenAI',
    'o3-mini': 'OpenAI',
    'o4-mini': 'OpenAI',
    # Google
    'gemini-1.5-pro': 'Google',
    'gemini-2.0-flash': 'Google',
    'gemini-2.5-flash': 'Google',
    'gemini-3-flash-preview': 'Google',
    'gemini-3-pro-preview': 'Google',
    'gemini-3.1-flash-lite-preview': 'Google',
    'gemini-3.1-pro-preview': 'Google',
    'gemma-2-27b-it-q8': 'Google',
    'gemma-2-9b-it': 'Google',
    'gemma-3-12b': 'Google',
    'gemma-3-27b': 'Google',
    'gemma-3-4b': 'Google',
    'gemma-3n-e4b-it': 'Google',
    'gemma-4-26b-a4b-it': 'Google',
    'gemma-4-31b-it': 'Google',
    # Meta
    'llama-3.1-405b': 'Meta',
    'llama-3.1-70b': 'Meta',
    'llama-3.1-8b': 'Meta',
    'llama-3.1-nemotron-70b-instruct': 'Meta',
    'llama-3.3-70b': 'Meta',
    'llama-4-scout': 'Meta',
    'llama-maverick': 'Meta',
    # Mistral
    'magistral-medium': 'Mistral',
    'magistral-small-2506': 'Mistral',
    'ministral-8b-instruct-2410': 'Mistral',
    'mistral-large-2411': 'Mistral',
    'mistral-large-2512': 'Mistral',
    'mistral-medium-2508': 'Mistral',
    'mistral-nemo-2407': 'Mistral',
    'mistral-saba': 'Mistral',
    'mistral-small-24b-instruct-2501': 'Mistral',
    'mistral-small-2506': 'Mistral',
    'mistral-small-2603': 'Mistral',
    'mistral-small-3.1-24b': 'Mistral',
    'mixtral-8x22b-instruct-v0.1': 'Mistral',
    'mixtral-8x7b-instruct-v0.1': 'Mistral',
    # DeepSeek
    'deepseek-chat-v3.1': 'DeepSeek',
    'deepseek-r1': 'DeepSeek',
    'deepseek-r1-0528': 'DeepSeek',
    'deepseek-r1-distill-llama-70b': 'DeepSeek',
    'deepseek-v3-0324': 'DeepSeek',
    'deepseek-v3-chat': 'DeepSeek',
    'DeepSeek-V3.2': 'DeepSeek',
    # Alibaba
    'qwen-3-8b': 'Alibaba',
    'qwen2-7b-instruct': 'Alibaba',
    'qwen2.5-32b-instruct': 'Alibaba',
    'qwen2.5-7b-instruct': 'Alibaba',
    'qwen2.5-coder-32b-instruct': 'Alibaba',
    'qwen3-30b-a3b': 'Alibaba',
    'qwen3-32b': 'Alibaba',
    'qwen3-coder-next': 'Alibaba',
    'qwen3-max-2025-09-23': 'Alibaba',
    'qwen3.5-35b-a3b': 'Alibaba',
    'qwen3.5-397b-a17b': 'Alibaba',
    'qwen3.6-plus': 'Alibaba',
    'qwq-32b': 'Alibaba',
    'Qwen3-Coder-480B-A35B-Instruct': 'Alibaba',
    # xAI
    'grok-3-mini-beta': 'xAI',
    'grok-4-fast': 'xAI',
    'grok-4.1-fast': 'xAI',
    'grok-4.20': 'xAI',
    # Amazon
    'command-a': 'Amazon',
    # Cohere
    'aya-expanse-32b': 'Cohere',
    'aya-expanse-8b': 'Cohere',
    'c4ai-command-r-08-2024': 'Cohere',
    # Microsoft
    'phi-3.5-mini-instruct': 'Microsoft',
    'phi-4': 'Microsoft',
    # Nvidia
    'llama-3.1-nemotron-70b-instruct': 'Nvidia',
    'nemotron-3-super-120b-a12b': 'Nvidia',
    # Autres
    'Apertus-70B-Instruct-2509': 'Mistral',
    'Apertus-8B-Instruct-2509': 'Mistral',
    'EuroLLM-22B-Instruct-2512': 'Autre',
    'Yi-1.5-9B-Chat': 'Autre',
    'chocolatine-14b-instruct-dpo-v1.2-q4': 'Autre',
    'chocolatine-2-14b-instruct-v2.0.3-q8': 'Autre',
    'glm-4.5': 'Autre',
    'glm-4.6': 'Autre',
    'glm-4.7': 'Autre',
    'glm-5': 'Autre',
    'glm-5.1': 'Autre',
    'hermes-3-llama-3.1-405b': 'Autre',
    'hermes-4-70b': 'Autre',
    'jamba-1.5-large': 'AI21 Labs',
    'kimi-k2': 'Autre',
    'kimi-k2-thinking': 'Autre',
    'kimi-k2.5': 'Autre',
    'kimi-k2.6': 'Autre',
    'lfm-40b': 'Autre',
    'lfm2-24b-a2b': 'Autre',
    'lfm2-8b-a1b': 'Autre',
    'minimax-m2': 'Autre',
    'minimax-m2.5': 'Autre',
    'minimax-m2.7': 'Autre',
    'olmo-3-32b-think': 'Autre',
    'trinity-large-preview': 'Autre',
}

# Vérification
non_mappes = [m for m in tous_modeles_sorted if m not in mapping_fournisseur]
print(f"Modèles non mappés : {len(non_mappes)}")
for m in non_mappes:
    print(f"  - {m}")

# Distribution par fournisseur
from collections import Counter
distrib = Counter(mapping_fournisseur.values())
for k, v in sorted(distrib.items(), key=lambda x: -x[1]):
    print(f"{k}: {v} modèles")

Modèles non mappés : 0
Autre: 23 modèles
OpenAI: 17 modèles
Mistral: 16 modèles
Google: 15 modèles
Alibaba: 14 modèles
DeepSeek: 7 modèles
Meta: 6 modèles
Anthropic: 5 modèles
xAI: 4 modèles
Cohere: 3 modèles
Nvidia: 2 modèles
Microsoft: 2 modèles
Amazon: 1 modèles
AI21 Labs: 1 modèles
